<a href="https://colab.research.google.com/github/markjschmidt/PET_Randoms/blob/main/MarkS_AnthropicAI_Random_Coinc_Events_20250105.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Simplify the final plots for Randoms

Claude:  Validating Random Coincidence Model in PET





Copy to final report.


My Prompt:

For plot 1,  Is that 372 time bins containing a pair of coincidence events or 372 singles events? What is the total number of bins in 1 second that have 2 events?

Can you replace "Time interval between zoomed coincidence events: 0.000000000 seconds" on the zoomed scatter plot with the time interval in nanoseconds?


Anthropic AI:

You're welcome! I apologize for any confusion. Let me clarify and make the requested changes.

The number 372 (or whatever number is printed) represents the total number of individual events that are part of a coincidence pair. Since each coincidence involves two events, the number of time bins containing a pair of coincidence events is actually half of this number.
To calculate the total number of bins in 1 second that have 2 events, we need to scale our result. Our simulation was for 1 millisecond, so we'll multiply by 1000 to get the number for 1 second.
I'll update the code to show the time interval in nanoseconds on the zoomed scatter plot.

Here's the updated code with these changes:


In [1]:
import numpy as np
import plotly.graph_objs as go
from scipy.stats import poisson

# Simulation parameters
total_events = 8000
time_resolution = 6e-9  # 6 nanoseconds
total_time = 1e-3  # 1 millisecond

# Generate random event times
event_times = np.random.uniform(0, total_time, total_events)

# Divide events into bins
num_bins = int(total_time // time_resolution)
event_counts, bin_edges = np.histogram(event_times, bins=num_bins, range=(0, total_time))

# Identify coincidence events (2 events in the same bin)
coincidence_bins = np.where(event_counts == 2)[0]

# Create arrays to store coincidence event times and colors
coincidence_times = []
marker_colors = ['blue'] * total_events

# Highlight coincidence events in red
for bin_index in coincidence_bins:
    bin_start, bin_end = bin_edges[bin_index], bin_edges[bin_index + 1]
    coincidence_events = event_times[(event_times >= bin_start) & (event_times < bin_end)]
    for event_time in coincidence_events:
        event_index = np.where(event_times == event_time)[0][0]
        marker_colors[event_index] = 'red'
        coincidence_times.append(event_time)

# Count red markers (coincidence events)
num_coincidence_events = marker_colors.count('red')
num_coincidence_bins = num_coincidence_events // 2
num_coincidence_bins_per_second = num_coincidence_bins * 1000  # Scale to 1 second

print(f"Number of coincidence events: {num_coincidence_events}")
print(f"Number of bins with coincidence events in 1 ms: {num_coincidence_bins}")
print(f"Number of bins with coincidence events in 1 second: {num_coincidence_bins_per_second}")

# Create scatter plot of all events
trace1 = go.Scatter(
    x=event_times,
    y=np.arange(1, total_events + 1),
    mode='markers',
    marker=dict(color=marker_colors, size=5),
    name='All Events'
)

# Plot full time range
fig1 = go.Figure(data=[trace1])
fig1.update_layout(
    title='Scatter Plot of Fluorine-18 Decay Events',
    xaxis_title='Time (s)',
    yaxis_title='Event Number',
    xaxis_range=[0, total_time],
    yaxis_range=[0, total_events + 1]
)
fig1.show()

# Find a pair of coincidence events for zoomed plot
if len(coincidence_times) >= 2:
    zoom_center = np.random.choice(coincidence_times)
    zoom_start = max(0, zoom_center - 300e-9)
    zoom_end = min(total_time, zoom_center + 300e-9)

    zoom_events = event_times[(event_times >= zoom_start) & (event_times <= zoom_end)]
    zoom_colors = [marker_colors[np.where(event_times == t)[0][0]] for t in zoom_events]

    trace_zoom = go.Scatter(
        x=zoom_events,
        y=np.arange(1, len(zoom_events) + 1),
        mode='markers',
        marker=dict(color=zoom_colors, size=10),
        name='Zoomed Events'
    )

    # Find and calculate time interval between coincidence events in zoom window
    zoom_coincidences = [t for t in coincidence_times if zoom_start <= t <= zoom_end]
    if len(zoom_coincidences) >= 2:
        interval_ns = abs(zoom_coincidences[1] - zoom_coincidences[0]) * 1e9  # Convert to nanoseconds
        interval_text = f"Time interval between zoomed coincidence events: {interval_ns:.2f} ns"
    else:
        interval_text = "No coincidence pair found in zoom window"

    fig_zoom = go.Figure(data=[trace_zoom])
    fig_zoom.update_layout(
        title='Zoomed Scatter Plot of Fluorine-18 Decay Events (600 ns)',
        xaxis_title='Time (s)',
        yaxis_title='Event Number',
        xaxis_range=[zoom_start, zoom_end],
        annotations=[
            dict(
                x=0.5,
                y=1.05,
                xref="paper",
                yref="paper",
                text=interval_text,
                showarrow=False,
            )
        ]
    )
    fig_zoom.show()

# Calculate Poisson probabilities for bin counts from 1 to 4
mu = total_events / num_bins
poisson_probs = [poisson.pmf(k, mu) for k in range(1, 5)]

# Create bar plot for Poisson distribution (1 to 4 events per bin)
trace2 = go.Bar(
    x=np.arange(1, 5),
    y=poisson_probs,
    marker_color='red',
    opacity=0.75
)

fig2 = go.Figure(data=[trace2])
fig2.update_layout(
    title='Poisson Distribution of Fluorine-18 Decay Events',
    xaxis_title='Number of Decay Events per Bin',
    yaxis_title='Probability',
    xaxis=dict(tickvals=[1, 2, 3, 4])
)
fig2.show()

print(f"Probability of 2 decay events per bin: {poisson_probs[1]:.6f}")


Number of coincidence events: 362
Number of bins with coincidence events in 1 ms: 181
Number of bins with coincidence events in 1 second: 181000


Probability of 2 decay events per bin: 0.001098


This updated code:

Calculates and prints the number of bins with coincidence events in both 1 ms and 1 second.
Shows the time interval between zoomed coincidence events in nanoseconds on the zoomed scatter plot.
Handles the case where no coincidence pair is found in the zoom window.

The output will now include:

The total number of coincidence events (individual events in red)
The number of bins with coincidence events in 1 ms
The estimated number of bins with coincidence events in 1 second
The probability of 2 decay events per bin

The zoomed scatter plot will display the time interval between coincidence events in nanoseconds (if a pair is found in the zoom window) or a message indicating that no coincidence pair was found.
This should provide a clearer picture of the frequency and distribution of coincidence events in your simulation.

